In [18]:
import requests
import pandas as pd
import time

df = pd.read_csv("communes.csv")

In [19]:
def get_data(lat, long,commune, page, cpt):
    BASE_URL = "https://recherche-entreprises.api.gouv.fr/near_point"
    params = {
        "lat" : lat,
        "long" : long, 
        "radius" : 5,
        "activite_principale" : "63.99Z,62.01Z",
        "per_page" : 25,
        "page" : page
    }
    
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if not data:
            return None,0
        colonnes = ['siren', 'nom_complet','categorie_entreprise','date_creation','date_fermeture','nombre_etablissements_ouverts','siege','dirigeants','tranche_effectif_salarie','siege.adresse','siege.coordonnees','finances']
        df_api = pd.json_normalize(data['results'])
        cols = [col for col in colonnes if col in df_api.columns]
        df_complet = df_api[cols]
        return df_complet,data.get('total_pages',1)
    else:
        print(response.status_code , " : PAGE NOT FOUND POUR COMMUNE :",commune)
        return None,0

In [20]:
debut = time.time()
resultat = pd.DataFrame()
cpt = 0
for i, row in df.iterrows():
    page = 1
    nb_page = 1
    while(page<=nb_page):
        cpt +=1
        df_temp, nb_page = get_data(row['latitude'],row['longitude'],row['commune'], page, cpt)
        if df_temp is not None and not df_temp.empty:
            resultat = pd.concat([resultat,df_temp])
        page +=1
    time.sleep(1)
resultat = resultat.drop_duplicates(subset=['siren'])
resultat.to_csv("entreprises.csv", index=False)
fin = time.time()
print(fin - debut)

400  : PAGE NOT FOUND POUR COMMUNE : VENOSC
400  : PAGE NOT FOUND POUR COMMUNE : AUTRANS
400  : PAGE NOT FOUND POUR COMMUNE : VEYRINS THUELLIN


KeyboardInterrupt: 